# Analytical Queries (Section C)
**NOTE:** Connected in READ_ONLY mode to allow multiple notebooks to be open.

In [1]:
import duckdb
import pandas as pd

# 1. Connect in READ ONLY mode to avoid file locking errors
try:
    con = duckdb.connect('../ecommerce.duckdb', read_only=True)
    print("Connected Successfully in Read-Only Mode.")
except Exception as e:
    print(f"Error connecting: {e}")

try:
    con.execute("PRAGMA threads=2")
    con.execute("PRAGMA memory_limit='10GB'")
    print("Stability Pragmas Applied.")
except Exception as e:
    print(f"Note: Could not set pragmas (already set): {e}")

queries = [
'''-- Query 1: Funnel Analysis
SELECT
  p.category_main,
  COUNT(DISTINCT CASE WHEN e.event_type='view'     THEN e.user_key END) AS viewers,
  COUNT(DISTINCT CASE WHEN e.event_type='cart'     THEN e.user_key END) AS carted,
  COUNT(DISTINCT CASE WHEN e.event_type='purchase' THEN e.user_key END) AS purchasers,
  ROUND(100.0 * COUNT(DISTINCT CASE WHEN e.event_type='cart' THEN e.user_key END)
    / NULLIF(COUNT(DISTINCT CASE WHEN e.event_type='view' THEN e.user_key END),0), 2)
    AS view_to_cart_pct,
  ROUND(100.0 * COUNT(DISTINCT CASE WHEN e.event_type='purchase' THEN e.user_key END)
    / NULLIF(COUNT(DISTINCT CASE WHEN e.event_type='cart' THEN e.user_key END),0), 2)
    AS cart_to_purchase_pct
FROM fact_events e
JOIN dim_product p ON e.product_key = p.product_key
GROUP BY p.category_main
ORDER BY viewers DESC
LIMIT 20;''',

'''-- Query 2: Session Aggregation
SELECT
  user_session,
  COUNT(*)                                            AS total_events,
  COUNT(DISTINCT product_key)                         AS distinct_products,
  SUM(CASE WHEN event_type='cart' THEN price ELSE 0 END) AS total_cart_value,
  MAX(CASE WHEN event_type='purchase' THEN 1 ELSE 0 END) AS had_purchase
FROM fact_events
GROUP BY user_session
ORDER BY total_events DESC
LIMIT 10;''',

'''-- Query 3: Top 10 Brands by Revenue per Month
WITH brand_revenue AS (
  SELECT
    e.event_month,
    p.brand,
    SUM(e.price) AS total_revenue,
    RANK() OVER (PARTITION BY e.event_month ORDER BY SUM(e.price) DESC) AS rnk
  FROM fact_events e
  JOIN dim_product p ON e.product_key = p.product_key
  WHERE e.event_type = 'purchase'
    AND p.brand IS NOT NULL
    AND p.brand != 'unknown'
  GROUP BY e.event_month, p.brand
)
SELECT event_month, brand, ROUND(total_revenue,2) AS total_revenue, rnk
FROM brand_revenue
WHERE rnk <= 10
ORDER BY event_month, rnk;''',

'''-- Query 4: October Buyers Who Did Not Return in November
SELECT DISTINCT oct.user_id
FROM (
  SELECT DISTINCT u.user_id
  FROM fact_events e
  JOIN dim_user u ON e.user_key = u.user_key
  WHERE e.event_type = 'purchase' AND e.event_month = 10
) oct
LEFT JOIN (
  SELECT DISTINCT u.user_id
  FROM fact_events e
  JOIN dim_user u ON e.user_key = u.user_key
  WHERE e.event_month = 11
) nov ON oct.user_id = nov.user_id
WHERE nov.user_id IS NULL
ORDER BY oct.user_id
LIMIT 10;''',

'''-- Query 5: Hourly Purchase Distribution
SELECT
  d.hour,
  COUNT(*) AS purchase_count,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_total
FROM fact_events e
JOIN dim_date d ON e.date_key = d.date_key
WHERE e.event_type = 'purchase'
GROUP BY d.hour
ORDER BY d.hour;'''
]

for i, q in enumerate(queries, 1):
    print(f"\n--- Running Query {i} ---")
    res = con.execute(q).df()
    display(res)


Connected Successfully in Read-Only Mode.
Stability Pragmas Applied.

--- Running Query 1 ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,category_main,viewers,carted,purchasers,view_to_cart_pct,cart_to_purchase_pct
0,electronics,3107670,640936,404255,20.62,63.07
1,uncategorized,2868261,333373,231804,11.62,69.53
2,appliances,1087757,168509,111151,15.49,65.96
3,apparel,625281,23279,13974,3.72,60.03
4,computers,547334,56799,35784,10.38,63.00
5,furniture,432393,20106,13293,4.65,66.11
6,auto,287456,20956,14835,7.29,70.79
7,kids,226188,10800,7710,4.77,71.39
8,construction,219134,18893,11354,8.62,60.10
9,accessories,115556,4063,2716,3.52,66.85



--- Running Query 2 ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

IOException: IO Error: Could not write file "C:\Users\jiyas\PycharmProjects\Jiya-Data-Engineering-Assignment-\ecommerce.duckdb.tmp\duckdb_temp_storage_S160K-0.tmp" (error in WriteFile): There is not enough space on the disk.
